<a href="https://colab.research.google.com/github/aburkov/theLMbook/blob/main/emotion_classifier_LR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="display: flex; justify-content: center;">
    <div style="background-color: #f4f6f7; padding: 15px; width: 80%;">
        <table style="width: 100%">
            <tr>
                <td style="vertical-align: middle;">
                    <span style="font-size: 14px;">
                        A notebook for <a href="https://www.thelmbook.com" target="_blank" rel="noopener">The Hundred-Page Language Models Book</a> by Andriy Burkov<br><br>
                        Code repository: <a href="https://github.com/aburkov/theLMbook" target="_blank" rel="noopener">https://github.com/aburkov/theLMbook</a>
                    </span>
                </td>
                <td style="vertical-align: middle;">
                    <a href="https://www.thelmbook.com" target="_blank" rel="noopener">
                        <img src="https://thelmbook.com/img/book.png" width="80px" alt="The Hundred-Page Language Models Book">
                    </a>
                </td>
            </tr>
        </table>
    </div>
</div>

### 5.3.1. Baseline Emotion Classifier (page 157)

In [26]:
# Import required libraries
import gzip             # For decompressing gzipped data files
import json             # For parsing JSON-formatted data
import random           # For shuffling dataset and setting seeds
import requests         # For downloading dataset from URL
from sklearn.feature_extraction.text import CountVectorizer # Text vectorization utility
from sklearn.linear_model import LogisticRegression         # Logistic regression model
from sklearn.metrics import accuracy_score                  # For model evaluation

# ----------------------------
# Utility Functions
# ----------------------------

def set_seed(seed):
    """
    Sets random seed for reproducibility.

    Args:
        seed (int): Seed value for random number generation
    """
    random.seed(seed)

def download_and_split_data(data_url, test_ratio=0.1):
    """
    Downloads emotion classification dataset from URL and splits into train/test sets.
    Handles decompression and JSON parsing of the raw data.

    Args:
        data_url (str): URL of the gzipped JSON dataset
        test_ratio (float): Proportion of data to use for testing (default: 0.1)

    Returns:
        tuple: (X_train, y_train, X_test, y_test) containing:
            - X_train, X_test: Lists of text examples for training and testing
            - y_train, y_test: Lists of corresponding emotion labels
    """
    # Download and decompress the dataset
    response = requests.get(data_url)
    content = gzip.decompress(response.content).decode()

    # Parse JSON lines into list of dictionaries
    dataset = [json.loads(line) for line in content.splitlines()]

    # Shuffle dataset for random split
    random.shuffle(dataset)

    # Split into train and test sets
    split_index = int(len(dataset) * (1 - test_ratio))
    train, test = dataset[:split_index], dataset[split_index:]

    # Separate text and labels
    X_train = [item["text"] for item in train]
    y_train = [item["label"] for item in train]
    X_test = [item["text"] for item in test]
    y_test = [item["label"] for item in test]

    return X_train, y_train, X_test, y_test

# ----------------------------
# Main Execution
# ----------------------------

# Set random seed for reproducibility
set_seed(42)

# Download and prepare dataset
data_url = "https://www.thelmbook.com/data/emotions"
X_train_text, y_train, X_test_text, y_test = download_and_split_data(
    data_url, test_ratio=0.1
)

print("Number of training examples:", len(X_train_text))
print("Number of test examples:", len(X_test_text))

# ----------------------------
# Baseline Model
# ----------------------------

# Initialize text vectorizer with basic parameters
# max_features=10_000: Limit vocabulary to top 10k most frequent words
# binary=True: Convert counts to binary indicators (0/1)
vectorizer = CountVectorizer(max_features=10_000, binary=True)

# Transform text data to numerical features
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Initialize and train logistic regression model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Make predictions on train and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate and display accuracy metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"\nTrain accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# ----------------------------
# Improved Model
# ----------------------------

print("\n--- Better hyperparameters ---")

# Initialize vectorizer with improved parameters
# max_features=20000: Increased vocabulary size
# ngram_range=(1, 2): Include both unigrams and bigrams
vectorizer = CountVectorizer(max_features=20000, ngram_range=(1, 2))

# Transform text data with new vectorizer
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

# Train and evaluate model with same parameters
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

Number of training examples: 18000
Number of test examples: 2000

Train accuracy: 0.9854
Test accuracy: 0.8875

--- Better hyperparameters ---
Train accuracy: 0.9961
Test accuracy: 0.8905


## Debug

### Peak into data

In [40]:
print(type(X_train_text), len(X_train_text))

<class 'list'> 18000


In [39]:
X_train_text[:4]

['i am not a professional historian by any means so some may feel as if i left out important things or took them out of context',
 'i somehow feel distraught and hopeless',
 'i feel like the nytimes publishes an article like this every year or so and each time we get pissed and feisty quick to lash out with a slew of offended and defensive responses',
 'i really hate that feeling when youre unsure about something']

In [18]:
y_train[:4]

['joy', 'fear', 'anger', 'fear']

In [19]:
print(len(X_test_text))
for i in X_test_text[:4]:
    print(i)

2000
i just feel so irritable which i guess is a classic symptom of depression
ive waited my whole life to feel this blessed now im comparing the dream to the way it is and everybodys looking there very best remembering times when they were just like this my imagination never felt so clear so no i know this is for real
im in a strange situation or feeling awkward i sometimes switch into comedian mode a bit of a defence mechanism from my self conscious school days and turned some of the sessions into katrinas minute stand up routine
four weeks ago i felt very much touched to find an asciatic patient who had asked the very morning to be tapped of the fluid


In [20]:
y_test[:4]

['anger', 'love', 'sadness', 'anger']

In [21]:
vectorizer

,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",20000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'


In [35]:
vectorizer.fit_transform(X_train_text[:4])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 130 stored elements and shape (4, 123)>

In [36]:
vectorizer.get_feature_names_out()

array(['about', 'about something', 'am', 'am not', 'an', 'an article',
       'and', 'and defensive', 'and each', 'and feisty', 'and hopeless',
       'any', 'any means', 'article', 'article like', 'as', 'as if', 'by',
       'by any', 'context', 'defensive', 'defensive responses',
       'distraught', 'distraught and', 'each', 'each time', 'every',
       'every year', 'feel', 'feel as', 'feel distraught', 'feel like',
       'feeling', 'feeling when', 'feisty', 'feisty quick', 'get',
       'get pissed', 'hate', 'hate that', 'historian', 'historian by',
       'hopeless', 'if', 'if left', 'important', 'important things',
       'lash', 'lash out', 'left', 'left out', 'like', 'like the',
       'like this', 'may', 'may feel', 'means', 'means so', 'not',
       'not professional', 'nytimes', 'nytimes publishes', 'of',
       'of context', 'of offended', 'offended', 'offended and', 'or',
       'or so', 'or took', 'out', 'out important', 'out of', 'out with',
       'pissed', 'pissed an

In [38]:
vectorizer.fit_transform(X_train_text[:4]).toarray()

array([[0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1,
        1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0,
        0, 1, 0, 1, 2, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 1, 3, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1,
        0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 0,

In [30]:
vectorizer.transform(X_test_text[:4])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 22 stored elements and shape (4, 123)>